# Flu Shot Learning - Notebook 12: repeated, stability-weighted ensemble

This is a new experiment after notebooks 1-11. It is designed for the mean ROC AUC leaderboard metric, not probability calibration. It uses repeated multilabel cross-validation, respondent-level interaction features, independent CatBoost and LightGBM learners, and a conservative blend selected by both OOF AUC and fold stability. The test set is never used to choose hyperparameters or blend weights.

Run every cell from top to bottom. The final file is `submission_notebook12_repeated_stable_blend.csv`.
+   

In [1]:
pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2026
N_SPLITS = 5
N_REPEATS = 2
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
MODEL_NAMES = ['catboost', 'lightgbm']
pd.set_option('display.max_columns', 100)

def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
        
    ]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError('Set DATA_DIR to the directory containing the four competition CSV files.')

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')
assert train_features['respondent_id'].is_unique
assert test_features['respondent_id'].is_unique
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert set(train_features['respondent_id']).isdisjoint(test_features['respondent_id'])
y = train_labels[TARGETS].astype('int8')
print(f'Data: {DATA_DIR.resolve()} | train={train_features.shape} | test={test_features.shape}')
display(y.mean().rename('positive_rate').to_frame())

Data: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset | train=(26707, 36) | test=(26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    behavior = [c for c in result if c.startswith('behavioral_')]
    opinions = [c for c in result if c.startswith('opinion_')]
    risk = [c for c in opinions if '_risk' in c]
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)
    result['behavior_score'] = result[behavior].sum(axis=1, min_count=1)
    result['behavior_available'] = result[behavior].notna().sum(axis=1)
    result['opinion_mean'] = result[opinions].mean(axis=1)
    result['opinion_risk_mean'] = result[risk].mean(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['recommendation_gap'] = result['doctor_recc_seasonal'] - result['doctor_recc_h1n1']
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)
    result['doctor_behavior_alignment'] = result['doctor_recommendation_count'] * result['behavior_score']
    result['doctor_opinion_alignment'] = result['doctor_recommendation_count'] * result['opinion_mean']
    result['behavior_opinion_alignment'] = result['behavior_score'] * result['opinion_mean']
    result['health_worker_insured'] = result['health_worker'] * result['health_insurance']
    result['age_education'] = result['age_group'].astype('string') + '_' + result['education'].astype('string')
    result['doctor_recommendation_pair'] = result['doctor_recc_h1n1'].astype('string') + '_' + result['doctor_recc_seasonal'].astype('string')
    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        result[f'{column}_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_columns]
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for column in categorical_columns:
    levels = pd.concat([X_lgb[column], X_test_lgb[column]], ignore_index=True).unique()
    dtype = pd.api.types.CategoricalDtype(categories=levels)
    X_lgb[column] = X_lgb[column].astype(dtype)
    X_test_lgb[column] = X_test_lgb[column].astype(dtype)
print(f'Features: {X.shape[1]} | categorical: {len(categorical_columns)}')

Features: 53 | categorical: 14


In [4]:
def make_catboost(target, seed):
    settings = {'h1n1_vaccine': (7, 2600, 0.025, 10), 'seasonal_vaccine': (6, 2600, 0.025, 12)}[target]
    depth, iterations, learning_rate, l2_leaf_reg = settings
    return CatBoostClassifier(
        depth=depth, iterations=iterations, learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg, loss_function='Logloss', eval_metric='AUC',
        random_seed=seed, verbose=False, allow_writing_files=False, thread_count=-1,
    )

def make_lightgbm(target, seed):
    return LGBMClassifier(
        objective='binary', n_estimators=3500, learning_rate=0.015,
        num_leaves=24 if target == 'h1n1_vaccine' else 31, max_depth=-1,
        min_child_samples=45, subsample=0.9, colsample_bytree=0.85,
        reg_alpha=0.2, reg_lambda=2.0, random_state=seed, n_jobs=-1, verbosity=-1,
    )
stratifier = MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = []
for repeat in range(N_REPEATS):
    splitter = MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE + repeat)
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y), start=1):
        folds.append((repeat, fold, train_idx, valid_idx))
print(f'Prepared {len(folds)} independent validation folds')

Prepared 10 independent validation folds


In [5]:
oof = {name: pd.DataFrame(0.0, index=X.index, columns=TARGETS) for name in MODEL_NAMES}
test_pred = {name: pd.DataFrame(0.0, index=X_test.index, columns=TARGETS) for name in MODEL_NAMES}
fold_scores = []

for repeat, fold, train_idx, valid_idx in folds:
    print(f'Repeat {repeat + 1}/{N_REPEATS}, fold {fold}/{N_SPLITS}')
    for target in TARGETS:
        seed = RANDOM_STATE + 1000 * repeat + fold
        cat = make_catboost(target, seed)
        cat.fit(X.iloc[train_idx], y.iloc[train_idx][target], cat_features=cat_feature_indices,
                eval_set=(X.iloc[valid_idx], y.iloc[valid_idx][target]),
                use_best_model=True, early_stopping_rounds=180)
        cat_valid = cat.predict_proba(X.iloc[valid_idx])[:, 1]
        cat_test = cat.predict_proba(X_test)[:, 1]
        oof['catboost'].iloc[valid_idx, oof['catboost'].columns.get_loc(target)] += cat_valid / N_REPEATS
        test_pred['catboost'][target] += cat_test / len(folds)

        lgb = make_lightgbm(target, seed + 100)
        lgb.fit(X_lgb.iloc[train_idx], y.iloc[train_idx][target],
                eval_set=[(X_lgb.iloc[valid_idx], y.iloc[valid_idx][target])],
                eval_metric='auc', categorical_feature=categorical_columns,
                callbacks=[early_stopping(180, verbose=False), log_evaluation(0)])
        lgb_valid = lgb.predict_proba(X_lgb.iloc[valid_idx])[:, 1]
        lgb_test = lgb.predict_proba(X_test_lgb)[:, 1]
        oof['lightgbm'].iloc[valid_idx, oof['lightgbm'].columns.get_loc(target)] += lgb_valid / N_REPEATS
        test_pred['lightgbm'][target] += lgb_test / len(folds)

        fold_scores.append({'repeat': repeat, 'fold': fold, 'target': target,
                            'catboost': roc_auc_score(y.iloc[valid_idx][target], cat_valid),
                            'lightgbm': roc_auc_score(y.iloc[valid_idx][target], lgb_valid)})

scores = pd.DataFrame(fold_scores)
display(pd.DataFrame({name: [roc_auc_score(y[target], oof[name][target]) for target in TARGETS] for name in MODEL_NAMES}, index=TARGETS).assign(mean=lambda d: d.mean(axis=1)))

Repeat 1/2, fold 1/5
Repeat 1/2, fold 2/5
Repeat 1/2, fold 3/5
Repeat 1/2, fold 4/5
Repeat 1/2, fold 5/5
Repeat 2/2, fold 1/5
Repeat 2/2, fold 2/5
Repeat 2/2, fold 3/5
Repeat 2/2, fold 4/5
Repeat 2/2, fold 5/5


,catboost,lightgbm,mean
h1n1_vaccine,0.872561,0.871307,0.871934
seasonal_vaccine,0.864820,0.864111,0.864466


In [6]:
def rank_normalize(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

def score_blend(weights, target, mode):
    if mode == 'rank':
        prediction = sum(weights[name] * rank_normalize(oof[name][target]) for name in MODEL_NAMES)
    else:
        prediction = sum(weights[name] * oof[name][target].to_numpy() for name in MODEL_NAMES)
    return roc_auc_score(y[target], prediction)

grid = np.arange(0.0, 1.01, 0.05)
best = {}
for target in TARGETS:
    candidates = []
    for cat_weight in grid:
        weights = {'catboost': float(cat_weight), 'lightgbm': float(1.0 - cat_weight)}
        for mode in ('probability', 'rank'):
            oof_score = score_blend(weights, target, mode)
            per_fold = []
            for _, group in scores[scores['target'] == target].groupby(['repeat', 'fold']):
                per_fold.append(weights['catboost'] * group['catboost'].iloc[0] + weights['lightgbm'] * group['lightgbm'].iloc[0])
            stability_adjusted = float(np.mean(per_fold) - 0.15 * np.std(per_fold))
            candidates.append((stability_adjusted, oof_score, mode, weights))
    _, oof_score, mode, weights = max(candidates, key=lambda item: (item[0], item[1]))
    best[target] = {'mode': mode, 'weights': weights, 'oof_auc': oof_score}
    print(target, best[target])
print('Selected mean OOF ROC AUC:', np.mean([best[target]['oof_auc'] for target in TARGETS]))

h1n1_vaccine {'mode': 'probability', 'weights': {'catboost': 1.0, 'lightgbm': 0.0}, 'oof_auc': np.float64(0.8725608872077936)}
seasonal_vaccine {'mode': 'probability', 'weights': {'catboost': 1.0, 'lightgbm': 0.0}, 'oof_auc': np.float64(0.8648198490897059)}
Selected mean OOF ROC AUC: 0.8686903681487497


In [7]:
final_predictions = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
for target in TARGETS:
    mode = best[target]['mode']
    weights = best[target]['weights']
    candidates = {name: (rank_normalize(test_pred[name][target]) if mode == 'rank' else test_pred[name][target].to_numpy()) for name in MODEL_NAMES}
    final_predictions[target] = sum(weights[name] * candidates[name] for name in MODEL_NAMES)

submission = pd.DataFrame({'respondent_id': test_features['respondent_id'], **{target: np.clip(final_predictions[target], 0.0, 1.0) for target in TARGETS}})
assert list(submission.columns) == ['respondent_id', *TARGETS]
assert submission['respondent_id'].equals(test_features['respondent_id'])
assert submission[TARGETS].notna().all().all()
assert ((submission[TARGETS] >= 0) & (submission[TARGETS] <= 1)).all().all()
output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission_notebook12_repeated_stable_blend.csv'
oof_path = output_dir / 'oof_notebook12_repeated_stable_blend.csv'
submission.to_csv(submission_path, index=False)
oof_frame = pd.DataFrame({'respondent_id': train_features['respondent_id']})
for name in MODEL_NAMES:
    for target in TARGETS:
        oof_frame[f'{name}_{target}'] = oof[name][target]
oof_frame.to_csv(oof_path, index=False)
print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())

Saved submission: /kaggle/working/outputs/submission_notebook12_repeated_stable_blend.csv
Saved OOF predictions: /kaggle/working/outputs/oof_notebook12_repeated_stable_blend.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.126580,0.178629
1,26708,0.023596,0.026151
2,26709,0.243320,0.718071
3,26710,0.678929,0.885962
4,26711,0.330408,0.479159
